### Import

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### All Collision Dataset

In [ ]:
gdf = gpd.read_file('data/cleaned/Collision_All_Filtered.geojson')
gdf

In [ ]:
sns.countplot(gdf, x='YEAR', color='steelblue')
plt.title('Seattle Collision Count by Year (2015-2025)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

sns.countplot(gdf[gdf['YEAR'] == 2025], x='MONTH', ax=ax, color='steelblue', label='2025', alpha=0.7)
sns.countplot(gdf[gdf['YEAR'] == 2024], x='MONTH', ax=ax, color='salmon', label='2024', alpha=0.7)
plt.title('Seattle Collision Count by Month in 2025')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

sns.countplot(gdf, x='Day_of_Week', order=day_order, color='steelblue')
plt.title('Total Collisions by Day of Week')
plt.xlabel('Day of the Week')
plt.ylabel('Number of crashes')
plt.xticks(rotation=45)
plt.show()

In [ ]:
order = ['Fall', 'Winter', 'Spring', 'Summer']
sns.countplot(gdf, x='Season', order=order, color='steelblue')
plt.title('Total Collisions Per Year by Season')
plt.xlabel('Season')
plt.ylabel('Total Crashes')
plt.show()

In [ ]:
gdf['TOTAL_PED'] = gdf['PEDCOUNT'] + gdf['PEDCYLCOUNT']
sns.lineplot(gdf, x='YEAR', y='TOTAL_PED', color='steelblue')
plt.title('Total Pedestrians Involved in Collisions by Year')
plt.xlabel('Year')
plt.ylabel('Total Pedestrians Involved')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 4))
hour_counts = gdf.groupby('HOUR').size().reset_index(name='COUNT')
sns.barplot(data=hour_counts, x='HOUR', y='COUNT', color='steelblue')
plt.title('Total Collisions by Hour of Day')
plt.xlabel('Hour of Day (0 = Midnight)')
plt.ylabel('Number of Collisions')
plt.tight_layout()
plt.show()

In [ ]:
heatmap_data = (
    gdf.groupby(['Day_of_Week', 'HOUR'])
    .size()
    .reset_index(name='COUNT')
)
heatmap_pivot = heatmap_data.pivot(index='Day_of_Week', columns='HOUR', values='COUNT')
heatmap_pivot = heatmap_pivot.reindex(day_order)

plt.figure(figsize=(16, 5))
sns.heatmap(heatmap_pivot, cmap='YlOrRd', linewidths=0.3, annot=False)
plt.title('Collision Frequency Heatmap: Day of Week vs. Hour of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
gdf_junc = gdf.groupby('JUNCTIONTYPE').size().sort_values(ascending=False)
sns.barplot(y=gdf_junc.index, x=gdf_junc.values, color='steelblue')
plt.title('Collision Count by Junction Type')
plt.tight_layout()
plt.show()

In [ ]:
gdf_filtered = gdf[gdf['YEAR'] >= 2015].copy()
gdf_filtered['PERIOD'] = gdf_filtered['YEAR'].apply(lambda y: 'Pre-2020' if y < 2020 else '2020 Onwards')

junc_period = (
    gdf_filtered.groupby(['PERIOD', 'JUNCTIONTYPE'])
    .size()
    .reset_index(name='COUNT')
)
junc_period['PROPORTION'] = junc_period.groupby('PERIOD')['COUNT'].transform(lambda x: x / x.sum())

plt.figure(figsize=(12, 5))
sns.barplot(data=junc_period, x='JUNCTIONTYPE', y='PROPORTION', hue='PERIOD', color='steelblue')
plt.title('Junction Type Distribution: Pre-2020 vs. 2020 Onwards (Proportional)')
plt.xlabel('Junction Type')
plt.ylabel('Proportion of Collisions')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
gdf_sev = gdf.groupby('SEVERITYDESC').size().sort_values(ascending=False)
sns.barplot(y=gdf_sev.index, x=gdf_sev.values, color='steelblue')
plt.title('Collision Count by Severity Description')
plt.tight_layout()
plt.show()

In [ ]:
severity_period = (
    gdf_filtered.groupby(['PERIOD', 'SEVERITYDESC'])
    .size()
    .reset_index(name='COUNT')
)
severity_period['PROPORTION'] = severity_period.groupby('PERIOD')['COUNT'].transform(lambda x: x / x.sum())

plt.figure(figsize=(12, 5))
sns.barplot(data=severity_period, x='SEVERITYDESC', y='PROPORTION', hue='PERIOD', color='steelblue')
plt.title('Severity Distribution: Pre-2020 vs 2020 Onwards (Proportional)')
plt.xlabel('Severity Description')
plt.ylabel('Proportion of Collisions')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
severity_year = (
    gdf.groupby(['YEAR', 'SEVERITYDESC'])
    .size()
    .reset_index(name='COUNT')
)
severity_year['TOTAL'] = severity_year.groupby('YEAR')['COUNT'].transform('sum')
severity_year['RATE'] = severity_year['COUNT'] / severity_year['TOTAL']

serious = severity_year[severity_year['SEVERITYDESC'].str.contains('Injury|Fatal', case=False, na=False)]

plt.figure(figsize=(12, 5))
sns.lineplot(data=serious, x='YEAR', y='RATE', hue='SEVERITYDESC', marker='o')
plt.title('Serious Outcome Rate Over Time (Proportion of All Collisions)')
plt.xlabel('Year')
plt.ylabel('Rate (Proportion)')
plt.xticks(sorted(serious['YEAR'].unique()), rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
severity_pivot = severity_year.pivot(index='YEAR', columns='SEVERITYDESC', values='RATE').fillna(0)

severity_pivot.plot(kind='area', stacked=True, figsize=(12, 5), alpha=0.75, colormap='tab10')
plt.title('Severity Proportion by Year (Stacked Area)')
plt.xlabel('Year')
plt.ylabel('Proportion of Collisions')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
gdf_sev = gdf.groupby('SDOT_COLDESC').size().sort_values(ascending=False).head(10).sort_values(ascending=True)
gdf_sev.plot(kind='barh')
plt.title('Collision Count by Description')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
gdf_sev = gdf.groupby('WEATHER').size().sort_values(ascending=True)
gdf_sev.plot(kind='barh')
plt.title('Collision Count by Weather')
plt.tight_layout()
plt.show()

In [ ]:
weather_sev = (
    gdf.groupby(['WEATHER', 'SEVERITYDESC'])
    .size()
    .reset_index(name='COUNT')
)
weather_sev = weather_sev[weather_sev['SEVERITYDESC'].str.contains('Injury|Fatal')]
weather_sev['PROPORTION'] = weather_sev.groupby('WEATHER')['COUNT'].transform(lambda x: x / x.sum())

plt.figure(figsize=(12, 5))
sns.barplot(data=weather_sev, x='WEATHER', y='PROPORTION', hue='SEVERITYDESC')
plt.title('Severity Proportion by Weather Condition')
plt.xlabel('Weather Condition')
plt.ylabel('Proportion of Collisions')
plt.xticks(rotation=30, ha='right')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
gdf_sev = gdf.groupby('ROADCOND').size().sort_values(ascending=False)
sns.barplot(y=gdf_sev.index, x=gdf_sev.values, color='steelblue')
plt.title('Collision Count by Road Condition')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
gdf_sev = gdf.groupby('LIGHTCOND').size().sort_values(ascending=False)
sns.barplot(y=gdf_sev.index, x=gdf_sev.values, color='steelblue')
plt.title('Collision Count by Light Condition')
plt.tight_layout()
plt.show()

### Vehicle Collision Dataset

In [ ]:
df_veh = pd.read_csv('data/cleaned/Vehicle_Filtered.csv')
df_veh

In [ ]:
df_veh_size = df_veh.groupby('ST_VEH_TYPE_DESC').size()
print(df_veh_size)

labels = np.unique(df_veh_size.index)
plt.figure(figsize=(10,10))
plt.pie(df_veh_size, labels=labels, autopct='%.2f%%', colors=sns.color_palette('muted', len(labels)))
plt.title('Distribution of Vehicle Types in Collisions')
plt.show()

In [ ]:
plt.figure(figsize=(10,5))
df_veh_group = pd.DataFrame(df_veh.groupby(['YEAR', 'ST_VEH_TYPE_DESC']).size().reset_index()).rename(columns={0: 'COUNT'})
df_veh_group = df_veh_group[df_veh_group['YEAR'] > 2018]

veh_plot = sns.barplot(data=df_veh_group, x='YEAR', y='COUNT', hue='ST_VEH_TYPE_DESC', legend=True)
sns.move_legend(
    veh_plot,
    'upper left',
    bbox_to_anchor=(1,1)
)
plt.tight_layout
plt.show()